# 📊 Exploratory Data Analysis (EDA) - Telco Customer Churn

**Objective:** Understand the dataset structure, distributions, patterns, and relationships before building the ML pipeline.

**Dataset:** Telco Customer Churn from Kaggle
- 7,043 customers
- 21 features
- Target: Churn (Yes/No)

## 1. Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

## 2. Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/telco_churn.csv')

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 3. Initial Data Inspection

In [ ]:
# First few rows
df.head(10)

In [ ]:
# Last few rows
df.tail()

In [ ]:
# Column names and data types
print("\n📋 Column Information:")
print("="*60)
df.info()

In [ ]:
# Shape and dimensions
print(f"\n📐 Dataset Shape: {df.shape}")
print(f"   Rows: {df.shape[0]:,}")
print(f"   Columns: {df.shape[1]}")

## 4. Missing Values Analysis

In [ ]:
# Check for missing values
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing_Count', ascending=False)

print("\n🔍 Missing Values Summary:")
print("="*60)
print(missing_data[missing_data['Missing_Count'] > 0])

if missing_data['Missing_Count'].sum() == 0:
    print("\n✅ No missing values found!")
else:
    print(f"\n⚠️ Total missing values: {missing_data['Missing_Count'].sum()}")

In [ ]:
# Check for empty strings or whitespace (common in CSV files)
print("\n🔍 Checking for empty strings or whitespace...")
for col in df.select_dtypes(include='object').columns:
    empty_count = (df[col] == '').sum()
    space_count = (df[col] == ' ').sum()
    if empty_count > 0 or space_count > 0:
        print(f"  {col}: {empty_count} empty strings, {space_count} whitespace")

## 5. Data Types and Unique Values

In [ ]:
# Data type summary
print("\n📊 Data Types Summary:")
print("="*60)
print(df.dtypes.value_counts())

In [ ]:
# Unique values per column
unique_summary = pd.DataFrame({
    'Column': df.columns,
    'Data_Type': df.dtypes,
    'Unique_Count': df.nunique(),
    'Sample_Values': [df[col].unique()[:3] for col in df.columns]
})

print("\n🔢 Unique Values Summary:")
print("="*60)
unique_summary

## 6. Identify Feature Types

In [ ]:
# Separate features by type
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove ID column if exists
if 'customerID' in categorical_cols:
    categorical_cols.remove('customerID')
    id_col = ['customerID']

# Identify target column
target_col = 'Churn'
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

print("\n📋 Feature Classification:")
print("="*60)
print(f"\n🔤 Categorical Features ({len(categorical_cols)}):")
print(categorical_cols)
print(f"\n🔢 Numerical Features ({len(numerical_cols)}):")
print(numerical_cols)
print(f"\n🎯 Target Variable: {target_col}")

## 7. Statistical Summary

In [ ]:
# Numerical features summary
print("\n📊 Numerical Features Statistics:")
print("="*60)
df[numerical_cols].describe().T

In [ ]:
# Categorical features summary
print("\n📊 Categorical Features Summary:")
print("="*60)
for col in categorical_cols[:5]:  # Show first 5
    print(f"\n{col}:")
    print(df[col].value_counts())
    print("-" * 40)

## 8. Target Variable Analysis

In [ ]:
# Target variable distribution
print("\n🎯 Target Variable (Churn) Distribution:")
print("="*60)
churn_counts = df['Churn'].value_counts()
churn_percentage = df['Churn'].value_counts(normalize=True) * 100

churn_summary = pd.DataFrame({
    'Count': churn_counts,
    'Percentage': churn_percentage.round(2)
})
print(churn_summary)

# Calculate imbalance ratio
if len(churn_counts) == 2:
    imbalance_ratio = churn_counts.max() / churn_counts.min()
    print(f"\n⚖️ Imbalance Ratio: {imbalance_ratio:.2f}:1")
    if imbalance_ratio > 2:
        print("⚠️ Dataset is imbalanced. Consider using SMOTE or class weights.")
    else:
        print("✅ Dataset is relatively balanced.")

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
df['Churn'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Add count labels
for i, v in enumerate(churn_counts):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', 
            startangle=90, colors=colors, textprops={'fontsize': 12})
axes[1].set_title('Churn Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Numerical Features Distribution

In [ ]:
# Distribution plots for numerical features
fig, axes = plt.subplots(len(numerical_cols), 2, figsize=(14, len(numerical_cols)*4))

for i, col in enumerate(numerical_cols):
    # Histogram
    axes[i, 0].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[i, 0].set_title(f'{col} - Distribution', fontsize=12, fontweight='bold')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Frequency')
    axes[i, 0].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
    axes[i, 0].axvline(df[col].median(), color='green', linestyle='--', label=f'Median: {df[col].median():.2f}')
    axes[i, 0].legend()
    
    # Box plot
    axes[i, 1].boxplot(df[col].dropna(), vert=False)
    axes[i, 1].set_title(f'{col} - Box Plot', fontsize=12, fontweight='bold')
    axes[i, 1].set_xlabel(col)

plt.tight_layout()
plt.show()

## 10. Categorical Features Distribution

In [ ]:
# Plot categorical features (first 8)
n_cols = 4
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols[:8]):
    value_counts = df[col].value_counts()
    axes[i].bar(range(len(value_counts)), value_counts.values, color='skyblue', edgecolor='black')
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[i].set_xticks(range(len(value_counts)))
    axes[i].set_xticklabels(value_counts.index, rotation=45, ha='right')
    axes[i].set_ylabel('Count')
    
    # Add count labels
    for j, v in enumerate(value_counts.values):
        axes[i].text(j, v + max(value_counts.values)*0.01, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 11. Correlation Analysis

In [ ]:
# Correlation matrix for numerical features
plt.figure(figsize=(10, 8))
correlation_matrix = df[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Correlation Matrix - Numerical Features', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Find highly correlated features (correlation > 0.8)
print("\n🔗 Highly Correlated Features (|correlation| > 0.8):")
print("="*60)
high_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.8:
            high_corr.append([
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ])

if high_corr:
    high_corr_df = pd.DataFrame(high_corr, columns=['Feature 1', 'Feature 2', 'Correlation'])
    print(high_corr_df)
else:
    print("✅ No highly correlated features found.")

## 12. Churn Analysis by Features

In [ ]:
# Churn rate by categorical features (first 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols[:6]):
    churn_by_feature = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    churn_by_feature.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'Churn Rate by {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Percentage (%)')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
    axes[i].legend(title='Churn', labels=['No', 'Yes'])

plt.tight_layout()
plt.show()

In [ ]:
# Numerical features vs Churn
fig, axes = plt.subplots(1, len(numerical_cols), figsize=(6*len(numerical_cols), 5))

for i, col in enumerate(numerical_cols):
    df.boxplot(column=col, by='Churn', ax=axes[i])
    axes[i].set_title(f'{col} by Churn')
    axes[i].set_xlabel('Churn')
    axes[i].set_ylabel(col)

plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.show()

## 13. Duplicate Records Check

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"\n🔍 Duplicate Records: {duplicates}")

if duplicates > 0:
    print(f"⚠️ Found {duplicates} duplicate rows")
    print("\nDuplicate rows:")
    print(df[df.duplicated(keep=False)])
else:
    print("✅ No duplicate records found")

# Check for duplicate customer IDs
if 'customerID' in df.columns:
    duplicate_ids = df['customerID'].duplicated().sum()
    print(f"\n🔍 Duplicate Customer IDs: {duplicate_ids}")
    if duplicate_ids > 0:
        print(f"⚠️ Found {duplicate_ids} duplicate customer IDs")
    else:
        print("✅ All customer IDs are unique")

## 14. Key Insights & Next Steps

In [ ]:
print("\n" + "="*70)
print("📝 KEY INSIGHTS & OBSERVATIONS")
print("="*70)

print("\n1️⃣ DATASET OVERVIEW:")
print(f"   • Total records: {len(df):,}")
print(f"   • Total features: {len(df.columns)}")
print(f"   • Categorical features: {len(categorical_cols)}")
print(f"   • Numerical features: {len(numerical_cols)}")

print("\n2️⃣ DATA QUALITY:")
print(f"   • Missing values: {df.isnull().sum().sum()}")
print(f"   • Duplicate records: {df.duplicated().sum()}")

print("\n3️⃣ TARGET VARIABLE:")
churn_rate = (df['Churn'] == 'Yes').mean() * 100
print(f"   • Churn rate: {churn_rate:.2f}%")
print(f"   • Class balance: {churn_summary['Percentage'].values}")

print("\n4️⃣ NEXT STEPS:")
print("   ✓ Check TotalCharges for data type issues (might need conversion)")
print("   ✓ Handle any empty strings in categorical features")
print("   ✓ Create feature engineering notebook")
print("   ✓ Test different encoding strategies for categorical variables")
print("   ✓ Consider feature interactions and new feature creation")
print("   ✓ Handle class imbalance if ratio > 2:1")
print("\n" + "="*70)

## 📊 Summary Statistics Export

In [ ]:
# Optional: Save summary statistics for reference
summary_stats = df.describe(include='all').T
summary_stats.to_csv('../reports/eda_summary_statistics.csv')
print("\n✅ Summary statistics saved to: reports/eda_summary_statistics.csv")